# Exploratory Data Analysis: Telco Customer Churn

Welcome to the exploratory data analysis for the Telco Customer Churn dataset. The main goal of this notebook is to understand the underlying patterns in our data before building any predictive models. By thoroughly exploring the features, we can identify which variables strongly correlate with customer churn and spot any data quality issues early on.

To ensure our analysis is robust and to avoid spilling information from the test set into our training process (a problem known as data leakage), we will split our dataset immediately after loading it. All the exploration, visualizations, and statistical summaries that follow will be performed solely on the training set. We will also use interactive Plotly charts with a clean, white background to make our findings easy to read and explore.

In [52]:
import numpy as np
import pandas as pd

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

import plotly.io as pio
pio.templates.default = 'plotly_white' # Enforce a clean white background for all plots

from sklearn.model_selection import train_test_split

import warnings
warnings.filterwarnings('ignore')

## Data Loading and Preparation

First, we load the raw dataset. A common issue with this specific dataset is that the `TotalCharges` column is sometimes parsed as text because of empty spaces for new customers. We handle this by coercing these empty strings into missing values (NaNs) and converting the column to a numeric format so we can safely compute statistics on it later.

We also drop the `customerID` column right away. While it uniquely identifies each customer, it does not hold any predictive power or useful information for our analysis.

After these initial cleaning steps, we divide the data into a training set (80%) and a test set (20%). We stratify the split based on the target variable (`Churn`) so that the proportion of churned customers remains consistent across both sets. Finally, we attach the target back to the training features to make our exploratory analysis easier.

In [53]:
# Load the initial dataset
df = pd.read_csv('../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv')

# Fix the TotalCharges formatting issue
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# Drop the customerID as it is not useful for analysis
if 'customerID' in df.columns:
    df = df.drop(columns=['customerID'])

# Split the data into training and test sets using stratification
X = df.drop('Churn', axis=1)
y = df['Churn']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

# Recombine the training features and target for exploration
df_train = X_train.copy()
df_train['Churn'] = y_train

print(f"Training dataset shape: {df_train.shape}")
df_train.head()

Training dataset shape: (5634, 20)


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
3738,Male,0,No,No,35,No,No phone service,DSL,No,No,Yes,No,Yes,Yes,Month-to-month,No,Electronic check,49.20,1701.65,No
3151,Male,0,Yes,Yes,15,Yes,No,Fiber optic,Yes,No,No,No,No,No,Month-to-month,No,Mailed check,75.10,1151.55,No
4860,Male,0,Yes,Yes,13,No,No phone service,DSL,Yes,Yes,No,Yes,No,No,Two year,No,Mailed check,40.55,590.35,No
3867,Female,0,Yes,No,26,Yes,No,DSL,No,Yes,Yes,No,Yes,Yes,Two year,Yes,Credit card (automatic),73.50,1905.70,No
3810,Male,0,Yes,Yes,1,Yes,No,DSL,No,No,No,No,No,No,Month-to-month,No,Electronic check,44.55,44.55,No


## Data Integrity Check: Missing Values and Cardinality

Before diving into visual patterns, it is crucial to understand the quality of our training data. We check for missing values to see if any imputation strategies are needed down the line. We also look at the number of unique values (cardinality) for each column. This helps us quickly distinguish between continuous numeric variables (which have many unique values) and categorical variables (which have a fixed, small set of possible values).

In [54]:
# Check for missing values in the training set
missing_values = df_train.isnull().sum()
print("Missing Values by Column:")
print(missing_values[missing_values > 0])

print("\nUnique Values count per Column (Variability):")
print(df_train.nunique())

Missing Values by Column:
TotalCharges    8
dtype: int64

Unique Values count per Column (Variability):
gender                 2
SeniorCitizen          2
Partner                2
Dependents             2
tenure                73
PhoneService           2
MultipleLines          3
InternetService        3
OnlineSecurity         3
OnlineBackup           3
DeviceProtection       3
TechSupport            3
StreamingTV            3
StreamingMovies        3
Contract               3
PaperlessBilling       2
PaymentMethod          4
MonthlyCharges      1489
TotalCharges        5275
Churn                  2
dtype: int64


## Exploring Categorical Features

Categorical variables describe qualities or groups (like gender, internet service type, or presence of dependents). To understand how these groups are distributed across our customer base, we will plot bar charts. By looking at the sheer count of customers in each category, we can identify which groups are the most common and spot any severe class imbalances.

We are arranging these bar charts into a grid of subplots so that we can scan the distributions across all categorical features at a glance.

In [55]:
# Isolate categorical columns (excluding our target variable 'Churn')
cat_cols = df_train.select_dtypes(include=['object']).columns.tolist()
if 'Churn' in cat_cols:
    cat_cols.remove('Churn')

# Determine the layout for the subplot grid
n_cols = 3
n_rows = int(np.ceil(len(cat_cols) / n_cols))

fig_cat = make_subplots(rows=n_rows, cols=n_cols, subplot_titles=cat_cols)

row = 1
col = 1
for c in cat_cols:
    # Count the occurrences of each category
    counts = df_train[c].value_counts().reset_index()
    counts.columns = [c, 'count']
    
    # Add a bar chart to the current subplot grid position
    fig_cat.add_trace(
        go.Bar(x=counts[c], y=counts['count'], name=c),
        row=row, col=col
    )
    
    col += 1
    if col > n_cols:
        col = 1
        row += 1

fig_cat.update_layout(height=300 * n_rows, width=1000, title_text="Training Set: Categorical Features Distribution", showlegend=False)
fig_cat.show()

## Exploring Numerical Features vs Churn

Numerical variables hold continuous mathematical values (like the number of months a customer has stayed or their monthly bill amount). To see if these continuous variables influence a customer's decision to leave, we compare their distributions between the group that churned and the group that did not.

For each numerical feature, we provide two perspectives:
- **Histograms** (top row): These show the shape and spread of the data. Overlaying the distributions helps us clearly see if one group tends to have higher or lower values than the other.
- **Box Plots** (bottom row): These provide a statistical summary. They clearly highlight the median, the interquartile range (the middle 50% of the data), and any extreme outliers.

In [56]:
# Identify the numerical columns
num_cols = df_train.select_dtypes(include=['int64', 'float64']).columns.tolist()

# Setup the subplot grid: 2 rows (Histograms on top, Box plots on the bottom) and one column for each numerical feature.
n_num = len(num_cols)
fig_num = make_subplots(
    rows=2, cols=n_num,
    subplot_titles=num_cols + [f'Box Plot: {c}' for c in num_cols]
)

# Define distinct colors for our target classes
colors = {'Yes': '#EF553B', 'No': '#636EFA'}

for i, c in enumerate(num_cols, 1):
    # Top Row: Overlayed Histograms
    for churn_val in ['No', 'Yes']:
        subset = df_train[df_train['Churn'] == churn_val]
        fig_num.add_trace(
            go.Histogram(x=subset[c], name=f'Churn={churn_val}', marker_color=colors[churn_val], opacity=0.75, legendgroup=churn_val, showlegend=(i==1)),
            row=1, col=i
        )
    
    # Bottom Row: Box Plots
    for churn_val in ['No', 'Yes']:
        subset = df_train[df_train['Churn'] == churn_val]
        fig_num.add_trace(
            go.Box(y=subset[c], x=subset['Churn'], name=f'Churn={churn_val}', marker_color=colors[churn_val], legendgroup=churn_val, showlegend=False),
            row=2, col=i
        )

# Use 'overlay' barmode so both Churn outcomes can be seen where they overlap in the histograms
fig_num.update_layout(barmode='overlay', height=700, width=1000, title_text="Distribution of Numerical Features Segmented by Churn")
fig_num.show()
